# Upload Documents to Azure Blob Storage

This notebook demonstrates how to upload PDF files and other documents from a local data folder to Azure Blob Storage using the Azure SDK for Python.

## Prerequisites

Before running this notebook, ensure you have:
1. An Azure Storage Account with Blob Storage enabled
2. A `.env` file with the following variables:
   - `BLOB_CONNECTION_STRING`: Your Azure Storage connection string
   - `BLOB_CONTAINER_NAME`: The name of the container to upload files to
3. Files to upload in the `data` folder

## Step 1: Import Required Libraries

We'll use the following libraries:
- `os` and `pathlib`: For file system operations
- `dotenv`: To load environment variables from `.env` file
- `azure.storage.blob`: Azure SDK for Blob Storage operations
- `azure.core.exceptions`: For handling Azure-specific exceptions

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient, BlobClient, ContainerClient
from azure.core.exceptions import ResourceExistsError

# Load environment variables from .env file
# Load environment variables from .env file in the root of workspace
load_dotenv(dotenv_path="../.env")


print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Step 2: Define Upload Function

This function handles uploading a single document to Azure Blob Storage. It:
1. Reads the connection string and container name from environment variables
2. Creates the container if it doesn't exist
3. Uploads the file with the option to overwrite existing blobs
4. Returns the URL of the uploaded blob

In [2]:
def upload_document_to_blob(file_path: str, blob_name: str = None):
    """
    Upload a document to Azure Blob Storage
    
    Args:
        file_path (str): Path to the file to upload
        blob_name (str): Name for the blob (defaults to filename)
    
    Returns:
        str: URL of the uploaded blob
    """
    # Get connection string and container name from .env
    connection_string = os.getenv("BLOB_CONNECTION_STRING")
    container_name = os.getenv("BLOB_CONTAINER_NAME")
    
    if not connection_string:
        raise ValueError("BLOB_CONNECTION_STRING not found in .env file")
    if not container_name:
        raise ValueError("BLOB_CONTAINER_NAME not found in .env file")
    
    # Use filename if blob_name not provided
    if blob_name is None:
        blob_name = Path(file_path).name
    
    try:
        # Create BlobServiceClient
        blob_service_client = BlobServiceClient.from_connection_string(connection_string)
        
        # Get container client
        container_client = blob_service_client.get_container_client(container_name)
        
        # Create container if it doesn't exist
        try:
            container_client.create_container()
            print(f"✓ Container '{container_name}' created")
        except ResourceExistsError:
            print(f"✓ Container '{container_name}' already exists")
        
        # Get blob client
        blob_client = blob_service_client.get_blob_client(
            container=container_name, 
            blob=blob_name
        )
        
        # Upload file
        print(f"\n📤 Uploading '{file_path}' to blob storage...")
        with open(file_path, "rb") as data:
            blob_client.upload_blob(data, overwrite=True)
        
        blob_url = blob_client.url
        print(f"✅ Successfully uploaded to: {blob_url}")
        
        return blob_url
        
    except FileNotFoundError:
        print(f"❌ Error: File '{file_path}' not found")
        raise
    except Exception as e:
        print(f"❌ Error uploading file: {str(e)}")
        raise

print("✅ Upload function defined!")

✅ Upload function defined!


## Step 3: Define List Blobs Function

This utility function lists all blobs currently stored in the container, showing their names and sizes.

In [3]:
def list_blobs_in_container():
    """List all blobs in the container"""
    connection_string = os.getenv("BLOB_CONNECTION_STRING")
    container_name = os.getenv("BLOB_CONTAINER_NAME")
    
    blob_service_client = BlobServiceClient.from_connection_string(connection_string)
    container_client = blob_service_client.get_container_client(container_name)
    
    print(f"\n📋 Blobs in container '{container_name}':")
    print("-" * 60)
    
    blob_list = container_client.list_blobs()
    for idx, blob in enumerate(blob_list, 1):
        print(f"{idx}. {blob.name} ({blob.size} bytes)")
    
    print("-" * 60)

print("✅ List blobs function defined!")

✅ List blobs function defined!


## Step 4: Define Batch Upload Function

This function uploads all files from a specified folder to Azure Blob Storage. It:
1. Validates the folder exists
2. Filters files by extension (optional)
3. Uploads each file and tracks success/failure counts

In [4]:
def upload_all_files_from_folder(folder_path: str, file_extensions: list = None):
    """
    Upload all files from a folder to Azure Blob Storage
    
    Args:
        folder_path (str): Path to the folder containing files
        file_extensions (list): List of file extensions to upload (e.g., ['.pdf', '.docx'])
                               If None, uploads all files
    
    Returns:
        tuple: (successful_uploads, failed_uploads)
    """
    if not os.path.exists(folder_path):
        print(f"❌ Error: Folder '{folder_path}' does not exist")
        return (0, 0)
    
    if not os.path.isdir(folder_path):
        print(f"❌ Error: '{folder_path}' is not a directory")
        return (0, 0)
    
    # Get all files in the folder
    all_files = []
    for item in os.listdir(folder_path):
        item_path = os.path.join(folder_path, item)
        if os.path.isfile(item_path):
            # Check file extension if specified
            if file_extensions is None or any(item.lower().endswith(ext) for ext in file_extensions):
                all_files.append(item_path)
    
    if not all_files:
        ext_msg = f" with extensions {file_extensions}" if file_extensions else ""
        print(f"❌ No files found in '{folder_path}'{ext_msg}")
        return (0, 0)
    
    print(f"\n📂 Found {len(all_files)} file(s) to upload from '{folder_path}'")
    
    successful = 0
    failed = 0
    
    for file_path in all_files:
        try:
            upload_document_to_blob(file_path)
            successful += 1
        except Exception as e:
            print(f"❌ Failed to upload '{file_path}': {str(e)}")
            failed += 1
    
    return (successful, failed)

print("✅ Batch upload function defined!")

✅ Batch upload function defined!


## Step 5: Configure and Run Upload

Now we'll configure the upload settings and execute the batch upload process.

### Configuration
- **Data Folder**: The script will look for files in the `data` folder
- **File Extensions**: Specify which file types to upload (PDF, DOCX, TXT, JSON, CSV)

In [5]:
print("=" * 80)
print("📄 Azure Blob Storage - Batch Upload Script")
print("=" * 80)

# Look for data folder in common locations
data_folder_paths = [
    "data",     # Current directory
    "../data",  # Parent directory
    "../input", # Alternative location
    "input"     # Alternative location
]

data_folder = None
for path in data_folder_paths:
    if os.path.exists(path) and os.path.isdir(path):
        data_folder = path
        break

if not data_folder:
    print(f"\n❌ Error: Could not find data folder in any of these locations:")
    for path in data_folder_paths:
        print(f"   - {os.path.abspath(path)}")
    print("\nPlease create a 'data' folder and place your files there.")
else:
    print(f"\n📁 Data folder found: {os.path.abspath(data_folder)}")
    
    # List files in the data folder
    files = os.listdir(data_folder)
    print(f"\n📋 Files in data folder:")
    for f in files:
        print(f"   - {f}")

📄 Azure Blob Storage - Batch Upload Script

📁 Data folder found: c:\Users\leepete\Documents\3.AI-Data\Development\16.workshop\Pfizer-AI-labs\Day3\HandsOnTasks\1.AISearch\data

📋 Files in data folder:
   - StudentLoanGuide.pdf
   - student_loan_discharge_guidance.pdf


## Step 6: Execute Batch Upload

Run the batch upload process to upload all matching files from the data folder to Azure Blob Storage.

In [6]:
# Define file extensions to upload
file_extensions = ['.pdf', '.docx', '.txt', '.json', '.csv']
print(f"📝 File types to upload: {', '.join(file_extensions)}")

# Execute batch upload
if data_folder:
    successful, failed = upload_all_files_from_folder(data_folder, file_extensions)
    
    print("\n" + "=" * 80)
    print(f"📊 Upload Summary:")
    print(f"   ✅ Successful: {successful}")
    print(f"   ❌ Failed: {failed}")
    print("=" * 80)
else:
    print("❌ Cannot proceed - data folder not found")

📝 File types to upload: .pdf, .docx, .txt, .json, .csv

📂 Found 2 file(s) to upload from 'data'
✓ Container 'content' already exists

📤 Uploading 'data\StudentLoanGuide.pdf' to blob storage...
✅ Successfully uploaded to: https://agentloanprocessing2025.blob.core.windows.net/content/StudentLoanGuide.pdf
✓ Container 'content' already exists

📤 Uploading 'data\student_loan_discharge_guidance.pdf' to blob storage...
✅ Successfully uploaded to: https://agentloanprocessing2025.blob.core.windows.net/content/student_loan_discharge_guidance.pdf

📊 Upload Summary:
   ✅ Successful: 2
   ❌ Failed: 0


## Step 7: Verify Uploaded Files

List all blobs in the container to verify the upload was successful.

In [7]:
# List all blobs in the container
list_blobs_in_container()

print("\n✅ Upload process complete!")


📋 Blobs in container 'content':
------------------------------------------------------------
1. 04268020-6fd0-43c8-9d4d-f0851bc07432/bs-JaneSmith_BankStatement.pdf (2771 bytes)
2. 04268020-6fd0-43c8-9d4d-f0851bc07432/la-JaneSmith-standard_rate-02.pdf (2935 bytes)
3. 10d91dd8-5d82-420e-b4b9-925e1ce1ca13/bs-JaneSmith_BankStatement.pdf (2771 bytes)
4. 10d91dd8-5d82-420e-b4b9-925e1ce1ca13/la-RobertWilson-reject-03.pdf (2909 bytes)
5. 1a1ac84e-b2b8-4bd5-b9fe-f6ae292a1dbc/bs-JaneSmith_BankStatement.pdf (2771 bytes)
6. 1a1ac84e-b2b8-4bd5-b9fe-f6ae292a1dbc/la-JaneSmith-standard_rate-02.pdf (2935 bytes)
7. 1e5f4154-88b4-46c4-9838-4a6def0db246/bs-JaneSmith_BankStatement.pdf (2851 bytes)
8. 1e5f4154-88b4-46c4-9838-4a6def0db246/la-JaneSmith-standard_rate-02.pdf (3015 bytes)
9. 284fe7dc-35c1-4e5b-9742-1820951ff88c/bs-JaneSmith_BankStatement.pdf (2771 bytes)
10. 284fe7dc-35c1-4e5b-9742-1820951ff88c/la-RobertWilson-reject-03.pdf (2909 bytes)
11. 33b67218-f883-48a8-a4e7-a84d200b2ca7/bs-JaneSmith_Bank

## Optional: Upload a Single File

Use this cell to upload a specific file manually.

In [ ]:
# Uncomment and modify the line below to upload a specific file
# upload_document_to_blob("data/your_file.pdf")